# 🎨 다이어트 및 자동화 완료 SigLIP 콜랩 (에러 핸들링 및 대시보드 강화)
- 생성한 벡터를 굳이 다운받지 않고 실시간으로 백엔드로 전송해 유저가 바로 체험 가능하도록 코드를 리팩토링했습니다.
- 중간에 에러가 뜨더라도 대시보드를 통해 원인을 알 수 있으며, `EMBEDDING_PROGRESS.md`로 구글 드라이브에 지속적으로 저장됩니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
WORK_DIR = '/content/drive/MyDrive/Armin_SigLIP_Project'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, 'public_data'), exist_ok=True)
print(f"✅ 구글 드라이브 마운트 및 폴더 준비 완료: {WORK_DIR}")
print("➡ 준비된 폴더에 search-manifest.json 와 공용 청크 데이터가 있는지 다시 한번 확인해주세요.")

In [ ]:
!pip install -q transformers sentencepiece torch pillow requests tqdm
import IPython
display(IPython.display.HTML('<h3>✅ 필수 라이브러리 설치 완료!</h3>'))

In [ ]:
import json, time, requests, urllib3, os, gc
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from io import BytesIO
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModel
from IPython.display import display, Markdown, clear_output
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ==============================================================
# ⚙️ 벡터 삽입 백엔드 URL (Cloudflare Worker)
WORKER_UPSERT_URL = "https://armin-semantic-search.armin-art.workers.dev/upsert"
BATCH_SIZE = 50  # 한번에 업로드할 묶음 개수
# ==============================================================

DATA_DIR = os.path.join(WORK_DIR, 'public_data')
OUTPUT_DIR = os.path.join(WORK_DIR, 'embedding_results')
STATE_FILE = os.path.join(WORK_DIR, 'siglip_state.json')
PROCESSED_FILE = os.path.join(WORK_DIR, 'siglip_processed_ids.txt')
DASHBOARD_FILE = os.path.join(WORK_DIR, 'EMBEDDING_PROGRESS.md')
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_ID = "google/siglip-base-patch16-224"

# 재시도 가능한 세션 설정 (이미지 다운로드 및 1회성 네트워크 오류 방지)
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
session.mount('http://', HTTPAdapter(max_retries=retries))
session.mount('https://', HTTPAdapter(max_retries=retries))

def load_state():
    state = { "stats": {"total_success": 0, "total_failed": 0, "total_upserted": 0}, "museum_counts": {}, "museum_processed": {} }
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r", encoding="utf-8") as f:
                state.update(json.load(f))
        except: pass
    processed_set = set()
    if os.path.exists(PROCESSED_FILE):
        try:
            with open(PROCESSED_FILE, "r", encoding="utf-8") as f:
                for line in f:
                    processed_set.add(line.strip())
        except: pass
    return state, processed_set

def save_state(state):
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f)

def render_dashboard(state, current_e=None, last_error=None):
    clear_output(wait=True)
    total_imgs = sum(state['museum_counts'].values())
    total_done = sum(state['museum_processed'].values())
    total_percent = (total_done / total_imgs * 100) if total_imgs > 0 else 0
    
    md_content = "# 🎨 SigLIP 임베딩 진행 현황 대시보드\n\n"
    md_content += f"> **마지막 업데이트**: {time.strftime('%Y-%m-%d %H:%M:%S')}\n"
    md_content += f"> **총 처리율**: **{total_percent:.2f}% ({total_done:,} / {total_imgs:,})**\n\n"
    md_content += f"- ✅ **성공(파일생성)**: {state['stats'].get('total_success',0):,} 건\n"
    md_content += f"- ☁️ **Cloudflare 업로드 완료**: {state['stats'].get('total_upserted',0):,} 건\n"
    md_content += f"- ❌ **실패(다운로드/네트워크 오류)**: {state['stats'].get('total_failed',0):,} 건\n\n"
    
    if last_error:
        md_content += f"⚠️ **최근 에러 원인**: `{last_error}`\n\n"
    if current_e and current_e != \"모두 완료 ��\":
        md_content += f"🔥 **현재 집중 처리 중인 영구전시**: `{current_e}`\n\n"
    elif current_e == \"모두 완료 🎉\":
        md_content += f"🎊 **모든 처리가 완료되었습니다!**\n\n"
        
    md_content += "| 영구전시 ID | 전체 수 | 완료 | 진행률(%) | 상태 |\n"
    md_content += "|:---|---:|---:|---:|:---:|\n"
    
    for e_id, total in sorted(state['museum_counts'].items()):
        done = state['museum_processed'].get(e_id, 0)
        percent = (done / total) * 100 if total > 0 else 0
        status = "✅ 완료" if done >= total and total > 0 else "⏳ 대기중"
        if current_e == e_id: status = "🔥 **진행중**"
        elif done > 0 and done < total: status = "⏳ 부분 진행"
        md_content += f"| **{e_id}** | {total:,} | {done:,} | {percent:.1f}% | {status} |\n"
        
    display(Markdown(md_content))
    try:
        with open(DASHBOARD_FILE, 'w', encoding='utf-8') as f:
            f.write(md_content)
    except: pass

def upload_to_cloudflare(batch):
    if not batch: return True
    payload = {"vectors": batch}
    try:
        resp = session.post(WORKER_UPSERT_URL, json=payload, timeout=40)
        if resp.status_code == 200: return True
        print(f"\n❌ Cloudflare Upload Error [{resp.status_code}]: {resp.text}")
        return False
    except Exception as e:
        print(f"\n❌ Cloudflare Network Error: {e}")
        return False

# GPU 및 모델 초기화
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 T4 GPU 가속 활성화됨: {device}")
model = AutoModel.from_pretrained(MODEL_ID).to(device)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()

# 데이터 및 진행상태 불러오기
state, processed_ids_set = load_state()
manifest_path = os.path.join(DATA_DIR, "search-manifest.json")
last_err_msg = None

if not os.path.exists(manifest_path):
    print("❌ 오류: 구글 드라이브에 search-manifest.json 파일이 없습니다! 업로드를 먼저 진행해주세요.")
else:
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    grouped_artworks = {}
    for chunk_file in manifest.get("chunks", []):
        chunk_path = os.path.join(DATA_DIR, chunk_file)
        if not os.path.exists(chunk_path): continue
        with open(chunk_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            artworks = data[0] if isinstance(data[0], list) else data
            for art in artworks:
                if not art.get("i"): continue
                e_id = art.get("e", "unknown")
                if e_id not in grouped_artworks: grouped_artworks[e_id] = []
                grouped_artworks[e_id].append(art)
                
    for e_id, arts in grouped_artworks.items():
        state["museum_counts"][e_id] = len(arts)
        
    output_file = os.path.join(OUTPUT_DIR, "siglip_embeddings.jsonl")
    sorted_exhibitions = sorted(grouped_artworks.keys())
    upload_batch = []
    
    render_dashboard(state)
    time.sleep(1)
    
    for e_id in sorted_exhibitions:
        arts = grouped_artworks[e_id]
        if state["museum_processed"].get(e_id, 0) >= len(arts): continue
        
        for idx, art in enumerate(arts):
            art_id = art.get("id") or f"{art.get('e','x')}-{art.get('n','x')}"
            img_url = art.get("i")
            if art_id in processed_ids_set: continue
            
            try:
                resp = session.get(img_url, timeout=12, verify=False, headers={"User-Agent": "Mozilla/5.0"})
                if resp.status_code == 200:
                    image = Image.open(BytesIO(resp.content)).convert("RGB")
                    inputs = processor(images=image, return_tensors="pt").to(device)
                    with torch.no_grad():
                        image_features = model.get_image_features(**inputs)
                    # 노말라이즈 (코사인 유사도 변환용)
                    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
                    vector = image_features[0].cpu().numpy().tolist()
                    
                    # 메모리 확보 (OOM 에러 방지)
                    del inputs, image_features, image
                    
                    # JSONL 저장 (백업)
                    result_record = {"id": art_id, "e": e_id, "vector": vector}
                    with open(output_file, "a", encoding="utf-8") as f_out:
                        f_out.write(json.dumps(result_record) + "\n")
                        
                    upload_batch.append({ "id": art_id, "values": vector, "metadata": {"e": e_id} })
                    state["stats"]["total_success"] = state["stats"].get("total_success", 0) + 1
                    state["museum_processed"][e_id] = state["museum_processed"].get(e_id, 0) + 1
                    last_err_msg = None
                    
                    # 50개 모이면 Cloudflare 전송
                    if len(upload_batch) >= BATCH_SIZE:
                        if upload_to_cloudflare(upload_batch):
                            state["stats"]["total_upserted"] = state.get("stats", {}).get("total_upserted", 0) + len(upload_batch)
                        upload_batch = []
                else:
                    state["stats"]["total_failed"] = state["stats"].get("total_failed", 0) + 1
                    last_err_msg = f"HTTP {resp.status_code} for URL 끝자리: {img_url[-20:]}"
            except Exception as e:
                state["stats"]["total_failed"] = state["stats"].get("total_failed", 0) + 1
                last_err_msg = f"에러({type(e).__name__}): {str(e)[:50]}"
                
            with open(PROCESSED_FILE, "a", encoding="utf-8") as f:
                f.write(art_id + "\n")
            processed_ids_set.add(art_id)
            
            # 20개 처리마다 상태 업데이트 및 메모리 정리
            if len(processed_ids_set) % 20 == 0:
                save_state(state)
                render_dashboard(state, current_e=e_id, last_error=last_err_msg)
                gc.collect()
                
    if upload_batch:
        if upload_to_cloudflare(upload_batch):
            state["stats"]["total_upserted"] = state.get("stats", {}).get("total_upserted", 0) + len(upload_batch)
            
    save_state(state)
    render_dashboard(state, current_e="모두 완료 🎉", last_error=None)
    print("\n🎊 모든 영구전시 SigLIP 임베딩 및 클라우드 업로드가 끝났습니다!")
